# Reproducibility: passports, citations and live data

The Zarr stores are **live datasets** — they grow as new releases land, so
"I used the ICOS data" is not reproducible. A reproducible claim names *which
slice* of *which release* was read *on what date*, and lets anyone check the
record wasn't altered. ICOS gives you that in three forms, one per access
route. This notebook demonstrates all three and verifies a passport's
integrity hash.

## 1. REST `/query` — the passport rides in the response

The passport's `query` field is the exact machine-readable request — anyone
can re-run it.

In [1]:
import io, json, requests
import pandas as pd

url = ("https://zarr.icos-cp.eu/query?id=icos-obspack.co2.co2"
       "&lat_min=51.9&lat_max=52.05&lon_min=4.85&lon_max=5.0"
       "&height_min=200&height_max=210"
       "&start=2023-06-01&end=2023-07-01&apply_qc=true&max_rows=100000")
*rows, last = requests.get(url).text.splitlines()
passport = json.loads(last)["_passport"]
rec = passport["@graph"][1]
print("rows:", rec["totalRows"], "| accessed:", rec["dateAccessed"])
print("query on record:", json.dumps(rec["query"][0], indent=1)[:400])

rows: 689 | accessed: 2026-08-17T17:14:47Z
query on record: {
 "variable": "co2",
 "store": "icos-obspack.zarr",
 "group": "co2",
 "query_kind": "station_time",
 "bbox": {
  "lat_min": 51.9,
  "lat_max": 52.05,
  "lon_min": 4.85,
  "lon_max": 5.0
 },
 "time": {
  "start": "2023-06-01",
  "end": "2023-07-01"
 },
 "height": {
  "min": 200.0,
  "max": 210.0
 },
 "apply_qc": true,
 "qc_rule": "icos-obspack ICOS ATC string flag; first char 'U' or 'O' = usable"



**Verify the integrity hash** — the passport self-certifies: null the
`passportSha256` field, canonicalise, hash, compare.

In [2]:
import copy, hashlib

p = copy.deepcopy(passport)
p["@graph"][1]["passportSha256"] = None
digest = hashlib.sha256(
    json.dumps(p, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
print("recomputed :", digest[:32], "…")
print("in passport:", rec["passportSha256"][:32], "…")
print("intact     :", digest == rec["passportSha256"])

recomputed : a3f483a131c6c19d52b01d8412d71caf …
in passport: a3f483a131c6c19d52b01d8412d71caf …
intact     : True


## 2. Direct Zarr reads — the session passport, on demand

Works for every store and domain; here the FLUXNET ecosystem store:

In [3]:
import xarray as xr

eco = xr.open_zarr("https://zarr.icos-cp.eu/icos-fluxnet.zarr/SE-Htm/fluxnet_mm",
                   consolidated=True)
_ = eco["NEE"].sel(ustar_threshold="VUT", nee_variant="REF").values   # real reads

r = requests.get(
    "https://zarr.icos-cp.eu/icos-fluxnet.zarr/session/passport").json()
rec2 = r["passport"]["@graph"][1]
print("session passport:", rec2["accessedGroups"], "|", rec2["dateAccessed"])
with open("fluxnet_session_passport.json", "w") as f:
    json.dump(r["passport"], f, indent=2)
print("saved → fluxnet_session_passport.json")

session passport: ['SE-Htm/fluxnet_mm'] | 2026-08-17T17:11:10Z
saved → fluxnet_session_passport.json


## 3. Portal objects — frozen releases with DOIs

Released objects never change; new releases point back via the version chain.
Store-level identity for the live Zarr stores comes from their Croissant
metadata:

In [4]:
cr = requests.get("https://zarr.icos-cp.eu/icos-fluxnet.zarr/croissant").json()
print("name:       ", cr.get("name"))
print("citeAs:     ", str(cr.get("citeAs"))[:120], "…")
print("DOI/url:    ", cr.get("url"))
print("live dataset:", cr.get("isLiveDataset"))
print("license:    ", cr.get("license"))

name:        icos-fluxnet
citeAs:      ICOS RI, Aalto, J., Aaltonen, H., Aiguier, T., Akubia, J., Alivernini, A., Allbrand, D., Aluome, C., Andersson, T., Arri …
DOI/url:     https://doi.org/10.18160/S6TB-567H
live dataset: True
license:     https://creativecommons.org/licenses/by/4.0/


## The checklist

Whichever route you used, your results folder should hold:

1. **the passport** (`/query` tail, arrow schema metadata, or
   `GET /<store>/session/passport`) — or, for portal objects, the object
   PID + citation;
2. **the access date** (in every passport; for live stores it is not
   optional);
3. **the citation** (in the passport / `citeAs` / `citationString`).

That is the difference between *repeatable* ("I ran some query again") and
*reproducible* ("this exact slice, this release, this date, this hash").